# QA output -> Word documents

Reads the per-figure `*_qa.pickle` files written by the three LMM test notebooks
and produces **one `.docx` per model**.

Layout, per the spec:

* a large header for each image name, each starting on a new page
* the full prompt the model was given -- persona, context, question and format --
  in **bold**
* the ground-truth answer
* *(blank line)*
* the LMM answer
* a horizontal rule between QA entries

## One thing to know before reading the output

The models reply with **two** fenced `json` blocks -- the answer, then a separate
explanation -- and the notebooks' parser keeps only the last one. So the stored
`Response` field is frequently the explanation *alone*, with the actual answer
dropped:

| model | entries | `Response` is explanation-only |
|---|---|---|
| chatgpt_api | 83 | 0 (0%) |
| claude_haiku | 83 | 82 (99%) |
| gemini | 83 | 56 (67%) |

Rendering `Response` directly would give a Claude document with essentially no
answers in it. This notebook therefore rebuilds the answer from `raw answer`,
merging **every** JSON object found there, so both the answer and its explanation
survive. `Response` is used only as a fallback when `raw answer` yields nothing.

This is a display-side repair -- it does not touch the pickles. The parser in
`llm_utils.parse_qa` is where it would be fixed properly.


In [1]:
# ============================ CONFIG ============================
# Where the test-run pickles live: one subdirectory per model.
out_base  = '~/Dropbox/WASP2026/LMM_outputs_tests/'

# One .docx per model lands here.
# docs_dir  = out_base + 'qa_word_docs/'
docs_dir = '~/SkyImagesWASP2026/models/test_models/word_docs_to_test/'

# Which model directories to convert.  None -> every subdirectory of out_base
# that contains at least one *_qa.pickle.
model_dirs = None
# model_dirs = ['chatgpt_api', 'claude_haiku', 'gemini']

# Include the model's explanation text alongside its answer.  The explanation is
# just another key in the merged answer, so False drops that one key.
include_explanation = True

# Show the question's difficulty level next to the question.  Off by default --
# the spec asks for the question alone.
include_level = False

# Show the FULL prompt -- persona + context + question + format -- rather than the
# bare question sentence.  This is the `Q` field, i.e. what the model was actually
# asked, minus the "I am going to show you an image" lead-in and the reasoning
# instruction, both of which are added at call time and are identical everywhere.
full_question = True

# Start each image on its own page.
page_break_per_image = True

verbose = True


In [2]:
import os
import re
import json
import glob
import pickle

from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# expand ~ ONCE and assign back -- glob('~/...') silently matches nothing
out_base = os.path.expanduser(out_base)
docs_dir = os.path.expanduser(docs_dir)

os.makedirs(docs_dir, exist_ok=True)

if not os.path.isdir(out_base):
    raise SystemExit('out_base does not exist: %s' % out_base)

if model_dirs is None:
    model_dirs = sorted(d for d in os.listdir(out_base)
                        if os.path.isdir(os.path.join(out_base, d))
                        and glob.glob(os.path.join(out_base, d, '*_qa.pickle')))

print('out_base :', out_base)
print('docs_dir :', docs_dir)
print('models   :', model_dirs)


out_base : /Users/jnaiman/Dropbox/WASP2026/LMM_outputs_tests/
docs_dir : /Users/jnaiman/SkyImagesWASP2026/models/test_models/word_docs_to_test/
models   : ['chatgpt_api', 'claude_haiku', 'gemini']


In [3]:
# ---------------------- answer recovery ----------------------

def json_objects(s):
    """
    Every top-level JSON object in `s`, in order.

    Uses the real JSON decoder rather than a regex: the model's replies contain
    nested braces and apostrophes, which a non-greedy brace pattern truncates.
    """
    if not isinstance(s, str):
        return []
    dec = json.JSONDecoder()
    out, i = [], 0
    while True:
        i = s.find('{', i)
        if i < 0:
            break
        try:
            obj, end = dec.raw_decode(s, i)
        except ValueError:
            i += 1          # not the start of a valid object; step past it
            continue
        if isinstance(obj, dict):
            out.append(obj)
        i = end
    return out


def recover_answer(entry):
    """
    The model's answer, reassembled.

    The models emit the answer and the explanation as two separate ```json
    blocks; `parse_qa` keeps only the last, so `entry['Response']` is usually
    the explanation by itself.  Merging every object from `raw answer` in order
    recovers the answer and leaves `explanation` as the final key.

    Returns (answer, lost) -- `lost` is True when the stored `Response` carried
    no answer key at all, i.e. rendering it directly would have shown an
    explanation with nothing to explain.  Observed in all three models, for
    different reasons: claude stores the explanation dict alone, gemini stores
    either that or the literal string "{", chatgpt stores the whole reply as an
    unparsed string.
    """
    merged = {}
    for obj in json_objects(entry.get('raw answer')):
        merged.update(obj)

    resp = entry.get('Response')
    if not merged:
        return resp, False                       # nothing to merge; use as-is
    if isinstance(resp, dict):
        for k, v in resp.items():
            merged.setdefault(k, v)              # keep anything only Response had

    # an "answer key" is any key other than the explanation
    stored_keys = set(resp) if isinstance(resp, dict) else set()
    lost = not (stored_keys - {'explanation'})
    return merged, lost


In [4]:
# ---------------------- formatting ----------------------

def fmt_scalar(v):
    """One value as display text.  Floats are kept readable, not exact."""
    if v is None:
        return '(none)'
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        return '%.6g' % v
    if isinstance(v, (list, tuple)):
        inner = [fmt_scalar(x) for x in v]
        return '[' + ', '.join(inner) + ']' if inner else '[]'
    if isinstance(v, dict):
        return '; '.join('%s: %s' % (k, fmt_scalar(x)) for k, x in v.items())
    s = str(v)
    return s if s.strip() else '(empty)'


def answer_lines(ans, drop_explanation=False):
    """An answer as a list of display lines -- one per key for a dict."""
    if isinstance(ans, dict):
        items = [(k, v) for k, v in ans.items()
                 if not (drop_explanation and k == 'explanation')]
        if not items:
            return ['(no answer)']
        if len(items) == 1 and items[0][0] != 'explanation':
            return [fmt_scalar(items[0][1])]      # single key: value alone reads better
        return ['%s: %s' % (k, fmt_scalar(v)) for k, v in items]
    return [fmt_scalar(ans)]


def add_hr(doc):
    """A horizontal rule.  python-docx has no HR, so use a paragraph bottom border."""
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(6)
    p.paragraph_format.space_after = Pt(6)
    pPr = p._p.get_or_add_pPr()
    borders = OxmlElement('w:pBdr')
    bottom = OxmlElement('w:bottom')
    bottom.set(qn('w:val'), 'single')
    bottom.set(qn('w:sz'), '6')
    bottom.set(qn('w:space'), '1')
    bottom.set(qn('w:color'), 'auto')
    borders.append(bottom)
    pPr.append(borders)
    return p


def add_labelled(doc, label, lines):
    """`label` in bold-italic, then the value lines -- first inline, rest below."""
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(0)
    run = p.add_run(label + ' ')
    run.bold = True
    run.italic = True
    p.add_run(lines[0])
    for extra in lines[1:]:
        q = doc.add_paragraph(extra)
        q.paragraph_format.space_after = Pt(0)
        q.paragraph_format.left_indent = Pt(18)
    return p


In [5]:
# ---------------------- build one document per model ----------------------

def image_name(pickle_path):
    """vqa_000003_qa.pickle -> vqa_000003"""
    stem = os.path.splitext(os.path.basename(pickle_path))[0]
    return stem[:-3] if stem.endswith('_qa') else stem


def build_doc(model_dir, verbose=True):
    src = os.path.join(out_base, model_dir)
    files = sorted(glob.glob(os.path.join(src, '*_qa.pickle')))
    if not files:
        print('[skip] no pickles in', src)
        return None

    doc = Document()
    model_id = None
    n_q = n_rec = 0
    first_image = True

    for pf in files:
        with open(pf, 'rb') as f:
            payload = pickle.load(f)
        # pickles are [qa_list, model_id_string]
        qa_list = payload[0] if isinstance(payload, (list, tuple)) else payload
        if isinstance(payload, (list, tuple)) and len(payload) > 1 and model_id is None:
            model_id = payload[1]

        # title goes in once we know the model id
        if doc.paragraphs == [] or not doc.paragraphs[0].text:
            title = doc.add_heading('%s -- QA responses' % model_dir, level=0)
            if model_id:
                sub = doc.add_paragraph()
                r = sub.add_run('model: %s' % model_id)
                r.italic = True

        head = doc.add_heading(image_name(pf), level=1)   # (1) large header per image
        # page_break_before on the heading itself, rather than an inserted break
        # paragraph -- no stray empty line at the foot of the previous page.
        # Skipped on the first heading so the title page isn't left blank.
        if page_break_per_image and not first_image:
            head.paragraph_format.page_break_before = True
        first_image = False

        for entry in qa_list:
            n_q += 1
            if full_question:
                # 'Q' is persona + context + question + format, already assembled
                # at QA-generation time.  Fall back to rebuilding it from the
                # parts, then to the bare question.
                question = entry.get('Q')
                if not question:
                    question = ' '.join(x for x in (entry.get('persona'),
                                                    entry.get('context'),
                                                    entry.get('question'),
                                                    entry.get('format')) if x)
            else:
                question = entry.get('question')
            question = ' '.join((question or '(no question text)').split())
            if include_level and entry.get('Level'):
                question = '[%s] %s' % (entry['Level'], question)

            # (2) the question, in bold
            p = doc.add_paragraph()
            p.paragraph_format.space_after = Pt(4)
            p.add_run(question).bold = True

            # (3) ground truth
            add_labelled(doc, 'Ground truth:', answer_lines(entry.get('A')))

            # blank line between (3) and (4)
            doc.add_paragraph()

            # (4) the LMM answer
            ans, lost = recover_answer(entry)
            n_rec += bool(lost)
            add_labelled(doc, 'LMM answer:',
                         answer_lines(ans, drop_explanation=not include_explanation))

            add_hr(doc)                              # rule between QA entries

    out = os.path.join(docs_dir, '%s_qa.docx' % model_dir)
    doc.save(out)
    if verbose:
        print('%-13s %2d figures, %3d questions (%d had no answer in the stored '
              'Response, rebuilt from raw) -> %s'
              % (model_dir, len(files), n_q, n_rec, out))
    return out


written = [build_doc(m, verbose=verbose) for m in model_dirs]
written = [w for w in written if w]
print()
print('wrote %d document(s)' % len(written))


chatgpt_api    3 figures,  67 questions (65 had no answer in the stored Response, rebuilt from raw) -> /Users/jnaiman/SkyImagesWASP2026/models/test_models/word_docs_to_test/chatgpt_api_qa.docx
claude_haiku   3 figures,  83 questions (82 had no answer in the stored Response, rebuilt from raw) -> /Users/jnaiman/SkyImagesWASP2026/models/test_models/word_docs_to_test/claude_haiku_qa.docx
gemini         3 figures,  83 questions (77 had no answer in the stored Response, rebuilt from raw) -> /Users/jnaiman/SkyImagesWASP2026/models/test_models/word_docs_to_test/gemini_qa.docx

wrote 3 document(s)


In [12]:
# ---------------------- spot-check ----------------------
# Print what went into the document for one figure, so the .docx can be checked
# against the source without opening Word.

check_model = model_dirs[0]
check_n     = 50         # how many QA entries to show
iqa = 2

pf = sorted(glob.glob(os.path.join(out_base, check_model, '*_qa.pickle')))[iqa]
qa_list = pickle.load(open(pf, 'rb'))[0]

print('=' * 70)
print('%s / %s' % (check_model, image_name(pf)))
print('=' * 70)
for entry in qa_list[:check_n]:
    ans, lost = recover_answer(entry)
    print()
    print('Raw Answer:')
    print(entry['raw answer'])
    print('Q: %s' % ((entry.get('Q') if full_question else entry.get('question')) or ''))
    for i, l in enumerate(answer_lines(entry.get('A'))):
        print('   %-13s %s' % ('ground truth:' if i == 0 else '', l[:300]))
    for i, l in enumerate(answer_lines(ans, drop_explanation=not include_explanation)):
        print('   %-13s %s' % ('LMM answer:' if i == 0 else '', l[:300]))
    if lost:
        stored = entry.get('Response')
        print('   [rebuilt from "raw answer" -- stored Response held only %s]'
              % (sorted(stored) if isinstance(stored, dict) else 'an unparsed %s' % type(stored).__name__))
    print('-' * 70)


chatgpt_api / vqa_000004

Raw Answer:
{"plot style":"classic"}
{"explanation":"The figure uses the default Matplotlib look: white/gray background with no ggplot-style gridlines, thin black axis spines, standard serif font, and conventional tick marks without the distinctive ggplot palette/grid. This matches the Matplotlib \"classic\" style rather than \"ggplot\"."}
Q: You are a helpful assistant that can analyze images. Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.
   ground truth: classic
   LMM answer:   plot style: classic
                 explanation: The figure uses the default Matplotlib look: white/gray background with no ggplot-style gridlines, thin black axis spines, standard serif font, and conventional tick marks without the distinctive ggplot palette/gr

In [11]:
pf

'/Users/jnaiman/Dropbox/WASP2026/LMM_outputs_tests/chatgpt_api/vqa_000001_qa.pickle'